<div style="background: linear-gradient(135deg, #0f2027, #203a43, #2c5364); padding: 28px; border-radius: 24px; text-align: center; color: #f2f5f7; box-shadow: 0 8px 20px rgba(0,0,0,0.12);">
  <h1 style="font-size: 42px; margin-bottom: 8px;">🌃 Seeing in the Dark 🔦</h1>
  <h2 style="font-size: 24px; margin-top: 0;">00 · Dataset Inventory — LoLI-Street &amp; ExDark</h2>
  <p style="font-size: 18px;">No downloads, no guesses: every number below is read straight from the local files.</p>
</div>

### Imports

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MANIFESTS = REPO / "data_manifests"
loli = json.loads((MANIFESTS / "loli_inventory.json").read_text())
exdark = json.loads((MANIFESTS / "exdark_inventory.json").read_text())
print("manifests loaded from", MANIFESTS)

<div style="background: linear-gradient(135deg, #74c69d, #48cae4); padding: 26px; border-radius: 22px; margin-top: 28px; border: 4px solid #2d6a4f; box-shadow: 0 8px 22px rgba(0, 0, 0, 0.18); color: #081c15;">
  <h1 style="color: #081c15; background-color: rgba(255,255,255,0.65); padding: 12px 16px; border-radius: 14px; margin-top: 0;">🛣️ Part 1: LoLI-Street (paired)</h1>
  <p style="font-size: 17px; line-height: 1.7; color: #133f30; font-weight: 500;">
    Paired low-light / well-exposed street images with YOLO detection labels.
    Home of all paired training (Parts A and B) and the only PSNR/SSIM ground truth.
  </p>
</div>

### Layout and pairing

In [ ]:
for split, info in loli["splits"].items():
    print(f"{split}: high={info['n_high']} low={info['n_low']} paired={info['paired']}")
print("Test:", loli["test"])
print("Sample sizes (high):", loli["splits"]["Train"]["sample_sizes_high"][:2])

### Label format and classes

In [ ]:
print("class_names:", len(loli["class_names"]), "entries (COCO 80)")
for key, info in loli["yolo"].items():
    top = sorted(info["class_hist"].items(), key=lambda kv: -int(kv[1]))[:5]
    print(f"{key}: files={info['n_files']} empty={info['n_empty']} top={top}")

names = loli["class_names"]
hist = Counter({names[int(k)]: int(v) for k, v in loli["yolo"]["train_high"]["class_hist"].items()})
top10 = hist.most_common(10)
plt.figure()
plt.bar([k for k, _ in top10], [v for _, v in top10])
plt.xticks(rotation=45, ha="right")
plt.title("LoLI-Street train_high: top-10 labeled classes")
plt.tight_layout()
plt.show()

<div style="background: linear-gradient(135deg, #232526, #414345); padding: 26px; border-radius: 22px; margin-top: 28px; border: 4px solid #0c0c0e; box-shadow: 0 8px 22px rgba(0, 0, 0, 0.18); color: #f2f2f2;">
  <h1 style="color: #f2f2f2; background-color: rgba(255,255,255,0.15); padding: 12px 16px; border-radius: 14px; margin-top: 0;">🌑 Part 2: ExDark (unpaired, real low-light)</h1>
  <p style="font-size: 17px; line-height: 1.7; color: #e6e6e6;">
    Real unpaired low-light images with bounding boxes but no clean reference.
    Zero-shot eval for every LoLI-Street model, CycleGAN's low-light domain, mAP-only metrics.
  </p>
</div>

### Annotation format and coverage

In [ ]:
print("images:", exdark["images"])
csv = exdark["csv"]
print("boxes:", csv["n_boxes"], "unique images:", csv["n_unique_images"])
print("bad coord rows:", csv["bad_coord_rows"])
print("on disk, missing in csv:", csv["on_disk_missing_in_csv"])
print("in csv, missing on disk:", csv["in_csv_missing_on_disk"])

plt.figure()
plt.bar(list(csv["class_hist"].keys()), list(csv["class_hist"].values()))
plt.title("ExDark: boxes per numeric class id (0-11, names TBD)")
plt.xlabel("class id")
plt.ylabel("boxes")
plt.tight_layout()
plt.show()

<div style="background: linear-gradient(135deg, #f093fb, #f5576c); padding: 26px; border-radius: 22px; margin-top: 28px; border: 4px solid #a01040; box-shadow: 0 8px 22px rgba(0, 0, 0, 0.18); color: #fff0f5;">
  <h1 style="color: #fff0f5; background-color: rgba(255,255,255,0.15); padding: 12px 16px; border-radius: 14px; margin-top: 0;">📋 Part 3: What this means for the project</h1>
  <p style="font-size: 17px; line-height: 1.7; color: #ffe0e8;">
    LoLI-Street labels use COCO 0-79 ids; ExDark uses its own 0-11 ids with no names file on disk,
    so class overlap must be resolved explicitly before any cross-dataset mAP comparison.
    LoLI-Street Test has no labels and stays qualitative-only.
  </p>
</div>

### Open items carried forward

- ExDark numeric class ids 0-11 need an external id-to-name mapping (confirm before converting to YOLO format).
- Cross-dataset mAP comparisons are restricted to overlapping classes until that mapping lands.
- Detection training uses the low-split labels where the input is a low-light image.